# Vanguard Signals API: validation

This notebook checks the Signals API end to end and ends with a pass/fail summary:

1. **Test suite:** offline unit and integration tests.
2. **Build outputs:** trip calibration, out-of-sample backtest, build-time public data.
3. **Live sources:** every feed responds.
4. **Cross-checks:** API output against each raw source, fetched independently.
5. **Schema and invariants:** shape, units, no earnings fields, API key.
6. **Public-records sanity:** coordinates, ridership weekday pattern, crash and ticket timing, curb data.
7. **Charts** for a visual read.

**Run it anywhere, with no deployment:**

- **Google Colab:** open this notebook, choose *Runtime → Run all*, and pick `vanguard-twin-nyc.zip` when the upload box appears.
- **Locally:** open it from inside the project folder (or next to the zip) and run all cells.

The setup cell unpacks the project, installs dependencies and runs both data builds. Expect 10–20 minutes on the first run, mostly downloading TLC trip records.

Checks that can't run, for example without network access, are marked `SKIP` or `WARN` rather than failing. A clean run should show no `FAIL`. Results are also written to `validation_report.json`.

## 0. Settings and setup

In [ ]:
# ---- Settings -------------------------------------------------------------
RUN_TRIP_BUILD   = True    # TLC trip calibration: ~1 GB download, 5-15 min. Skipped if already built.
RUN_EXTRAS_BUILD = True    # subway ridership, crashes, tickets, curb: a few minutes. Skipped if already built.
FORCE_REBUILD    = False   # rebuild even if outputs exist
NWS_CONTACT      = "vanguard-twin-nyc (Kartick Rajen)"   # the Weather Service asks for an app name / contact
TICKETMASTER_API_KEY = ""  # optional: enables the event checks. Leave blank before sharing this notebook.
# ---------------------------------------------------------------------------

import os, sys, glob, zipfile, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Setting up. If an upload box appears below, choose vanguard-twin-nyc.zip.", flush=True)

def find_project():
    for base in [Path.cwd(), Path.cwd().parent, Path("/content")]:
        for cand in [base, base / "vanguard-twin-nyc"]:
            if (cand / "vanguard" / "signals.py").exists():
                return cand
    return None

# Unpack the newest vanguard-twin-nyc*.zip found here (code is refreshed, built data/ is kept)
zips = sorted(set(glob.glob("vanguard-twin-nyc*.zip") + glob.glob("/content/vanguard-twin-nyc*.zip")),
              key=os.path.getmtime)
if not zips and find_project() is None and IN_COLAB:
    from google.colab import files
    print("Waiting for vanguard-twin-nyc.zip. If no upload box shows, stop this cell, drag the zip into "
          "the Files panel on the left, and run the cell again.", flush=True)
    zips = [name for name in files.upload() if name.endswith(".zip")]
if zips:
    target = Path(zips[-1]).resolve().parent
    with zipfile.ZipFile(zips[-1]) as z:
        for m in z.namelist():
            if "/data/" in m and not m.endswith((".gitkeep", "/")):
                continue          # never overwrite built data
            z.extract(m, target)
    print("unpacked", zips[-1])
ROOT = find_project()
if ROOT is None:
    raise SystemExit("Project not found: put vanguard-twin-nyc.zip next to this notebook, or open it inside the project folder.")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("project:", ROOT)

# Dependencies: install only what's missing
def missing(mods):
    out = []
    for m in mods:
        try:
            __import__(m)
        except ImportError:
            out.append(m)
    return out

need = missing(["flask", "duckdb", "numpy", "dotenv", "requests", "pytest", "pandas", "matplotlib", "tzdata"])
if need:
    print("installing:", ", ".join(need), "(about a minute)", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt",
                    "-r", "requirements-notebook.txt"], check=True)

# Settings -> environment (read by vanguard.config at import, so this runs before any vanguard import)
if NWS_CONTACT:
    os.environ.setdefault("NWS_USER_AGENT", NWS_CONTACT)
if TICKETMASTER_API_KEY:
    os.environ["TICKETMASTER_API_KEY"] = TICKETMASTER_API_KEY

def run_step(label, args):
    # Run a build step, streaming its output; print a heartbeat when it goes quiet
    import threading, time
    print(f"\n=== {label}: python {' '.join(args)}  (started {time.strftime('%H:%M:%S')})", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen([sys.executable, "-u", *args], cwd=ROOT, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    lines, last, t0 = [], [time.time()], time.time()

    def heartbeat():
        while proc.poll() is None:
            time.sleep(15)
            if proc.poll() is None and time.time() - last[0] > 60:
                print(f"    ... still running ({(time.time() - t0) / 60:.0f} min). Large downloads and queries go quiet; this is normal.", flush=True)
                last[0] = time.time()
    threading.Thread(target=heartbeat, daemon=True).start()

    for line in proc.stdout:
        line = line.rstrip()
        last[0] = time.time()
        if not line.strip():
            continue
        if "% of" in line and lines and "% of" in lines[-1]:   # download progress: print every ~10%
            pct = float(line.split("%")[0].split()[-1])
            prev = float(lines[-1].split("%")[0].split()[-1])
            lines[-1] = line
            if int(pct // 10) == int(prev // 10) and pct < 100:
                continue
        else:
            lines.append(line)
        print(line, flush=True)
    proc.wait()
    print(f"=== {label}: exit code {proc.returncode} after {(time.time() - t0) / 60:.1f} min", flush=True)
    return {"step": label, "returncode": proc.returncode, "tail": "\n".join(lines[-5:])}

BUILD_LOG = []
if RUN_TRIP_BUILD and (FORCE_REBUILD or not (ROOT / "data" / "baseline.json").exists()):
    BUILD_LOG.append(run_step("trip build", ["-m", "vanguard.build"]))
def extras_incomplete():
    f = ROOT / "data" / "extras.json"
    if not f.exists():
        return True
    try:
        import json as _json
        return any(v["status"] != "ok" for v in _json.loads(f.read_text())["meta"]["sections"].values())
    except Exception:
        return True

if RUN_EXTRAS_BUILD and (FORCE_REBUILD or extras_incomplete()):
    BUILD_LOG.append(run_step("extras build", ["-m", "vanguard.extras"]))
print("\nsetup done")

In [ ]:
import json, os, subprocess, sys, time, statistics
from datetime import datetime, timedelta
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import requests
from vanguard import config, feeds, model, signals, socrata, zones as zmod
from vanguard.stateplane import to_latlon
import app as webapp

try:
    import pandas as pd
    import matplotlib.pyplot as plt
    HAVE_PLOTS = True
except ImportError:
    HAVE_PLOTS = False
    print("pandas/matplotlib not installed: tables and charts are skipped (pip install -r requirements-notebook.txt)")

RESULTS = []

def check(section, name, status, detail=""):
    # status: PASS | WARN | FAIL | SKIP
    RESULTS.append({"section": section, "check": name, "status": status, "detail": str(detail)[:300]})
    print(f"[{status:4}] {section} / {name}" + (f": {detail}" if detail else ""))

def expect(section, name, cond, detail="", fail="FAIL"):
    check(section, name, "PASS" if cond else fail, detail)

print("python:", sys.version.split()[0], "| run at", datetime.now().isoformat(timespec="seconds"))

## 1. Test suite

In [ ]:
t0 = time.time()
proc = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"], cwd=ROOT,
                      capture_output=True, text=True)
tail = (proc.stdout.strip().splitlines() or ["(no output)"])[-1]
print(proc.stdout[-1500:])
expect("tests", "pytest suite", proc.returncode == 0, f"{tail} in {time.time() - t0:.0f}s")

## 2. Build outputs

In [ ]:
S = "build"
for b in BUILD_LOG:
    check(S, f"ran {b['step']}", "PASS" if b["returncode"] == 0 else "WARN", b["tail"].splitlines()[-1] if b["tail"] else "")
model.CAL.reload()
cal = model.CAL.status()
if cal["mode"] != "calibrated":
    check(S, "trip calibration", "WARN", "placeholder priors: run `python -m vanguard.build`")
else:
    check(S, "trip calibration", "PASS", cal["message"])
    bt = model.CAL.backtest
    d, p = bt["holdout"]["demand"], bt["holdout"]["pay"]
    expect(S, "weather lowers held-out trip error", (d.get("error_reduction_pct") or 0) > 0,
           f"{d.get('error_reduction_pct')}% overall", fail="WARN")
    if d.get("rain_error_reduction_pct") is None:
        check(S, "weather lowers trip error in rain hours", "SKIP", "no rain hours in the holdout")
    else:
        expect(S, "weather lowers trip error in rain hours", d["rain_error_reduction_pct"] > 0,
               f"{d['rain_error_reduction_pct']}% in {bt['holdout']['holdout_rain_zone_hours']} rain zone-hours", fail="WARN")
    coefs = {c["name"]: c for c in bt["final_fit"]["demand"]["coefs"]}
    ci = coefs["precip_mm"].get("ci95")
    expect(S, "rain coefficient significant", bool(ci) and (ci[0] > 0 or ci[1] < 0),
           f"beta={coefs['precip_mm']['beta']:.4f}, 95% CI={ci}", fail="WARN")

ex = signals.EXTRAS.get()
if not ex:
    check(S, "build-time public data", "WARN", "data/extras.json missing: run `python -m vanguard.extras`")
else:
    # Server outages and timeouts on the open-data portals are the portal's problem, not the code's
    NET = ("ConnectionError", "ProxyError", "Timeout", "needs the subway section",
           "429", "500 Server", "502", "503", "504")
    for name, st in ex["meta"]["sections"].items():
        err = st.get("error") or ""
        status = "PASS" if st["status"] == "ok" else ("WARN" if any(k in err for k in NET) else "FAIL")
        check(S, f"extras: {name}", status, (err + (" (server-side; re-run this notebook later)" if status == "WARN" else "")) if err else f"{st.get('seconds')}s")
    age = datetime.now() - datetime.fromisoformat(ex["meta"]["generated_at"])
    expect(S, "extras are recent", age < timedelta(days=45), f"built {age.days} days ago", fail="WARN")

## 3. Live sources

In [ ]:
S = "live"
LIVE = {}
fetchers = {"weather": feeds.weather, "traffic": feeds.traffic, "airports": feeds.airports, "events": feeds.events,
            "nws_alerts": feeds.nws_alerts, "transit_alerts": feeds.transit_alerts,
            "permitted_events": feeds.permitted_events, "street_closures": feeds.street_closures,
            "citibike": feeds.citibike}
for name, fn in fetchers.items():
    t0 = time.time()
    LIVE[name] = v = fn()
    st, secs = v.get("status"), time.time() - t0
    if st == "live":
        check(S, name, "PASS", f"{secs:.1f}s")
    elif st == "disabled":
        check(S, name, "SKIP", v.get("error"))
    else:
        check(S, name, "WARN", f"{st}: {v.get('error')}")
ONLINE = sum(v.get("status") == "live" for v in LIVE.values())
print(f"\n{ONLINE}/{len(LIVE)} sources live")
if not config.NWS_USER_AGENT or "set NWS_USER_AGENT" in config.NWS_USER_AGENT:
    check(S, "NWS_USER_AGENT set", "WARN", "set an app name and contact email in .env")
else:
    check(S, "NWS_USER_AGENT set", "PASS")

## 4. Cross-checks against raw sources
Each check fetches the raw source again, independently of the app's parsing, and compares.

In [ ]:
S = "cross-check"
SIG = signals.build(LIVE, hours=24)
UA = {"User-Agent": config.NWS_USER_AGENT}

def skip_if_offline(name, feed):
    if LIVE[feed].get("status") != "live":
        check(S, name, "SKIP", f"{feed} not live")
        return True
    return False

# Weather: API hour 0 equals the raw Open-Meteo hourly value
if not skip_if_offline("weather matches Open-Meteo", "weather"):
    try:
        raw = requests.get("https://api.open-meteo.com/v1/forecast", params={
            "latitude": config.NYC_LAT, "longitude": config.NYC_LON, "hourly": "temperature_2m,precipitation",
            "forecast_days": 2, "temperature_unit": "fahrenheit", "timezone": config.TZ}, timeout=10).json()
        key = SIG["city_hours"][0]["t"][:13]
        i = [t[:13] for t in raw["hourly"]["time"]].index(key)
        w = SIG["city_hours"][0]["weather"]
        ok = abs(w["temp_f"] - raw["hourly"]["temperature_2m"][i]) < 1.5 and abs((w["precip_mm"] or 0) - (raw["hourly"]["precipitation"][i] or 0)) < 0.5
        expect(S, "weather matches Open-Meteo", ok, f"{key}: api {w['temp_f']}°F / raw {raw['hourly']['temperature_2m'][i]}°F")
    except Exception as e:
        check(S, "weather matches Open-Meteo", "WARN", repr(e))

# Traffic: borough medians recomputed from raw DOT rows
if not skip_if_offline("traffic medians match NYC DOT", "traffic"):
    try:
        rows = socrata.get(socrata.NYC, "i4gi-tjb9", {"$limit": 400, "$order": "data_as_of DESC"})
        by = {}
        for r in rows:
            try:
                s = float(r.get("speed", 0))
            except ValueError:
                continue
            if r.get("borough") and 1 < s < 70:
                by.setdefault(r["borough"], []).append(s)
        raw_med = {b: statistics.median(v) for b, v in by.items() if len(v) >= 3}
        api_med = LIVE["traffic"]["median_mph"]
        diffs = {b: round(abs(api_med[b] - raw_med[b]), 1) for b in api_med if b in raw_med}
        # The feed refreshes every few minutes, so two samples taken apart can differ a little
        expect(S, "traffic medians match NYC DOT", diffs and max(diffs.values()) < 5, f"abs diff mph by borough {diffs}", fail="WARN")
    except Exception as e:
        check(S, "traffic medians match NYC DOT", "WARN", repr(e))

# NWS: every alert in the API is in the raw statewide list
if not skip_if_offline("NWS alerts are real NWS alerts", "nws_alerts"):
    try:
        raw = requests.get("https://api.weather.gov/alerts/active", params={"area": "NY"},
                           headers={**UA, "Accept": "application/geo+json"}, timeout=15).json()
        raw_ids = {f["properties"]["id"] for f in raw.get("features", [])}
        api_ids = {a["id"] for a in LIVE["nws_alerts"]["alerts"]}
        expect(S, "NWS alerts are real NWS alerts", api_ids <= raw_ids,
               f"{len(api_ids)} NYC alerts, {len(raw_ids)} statewide, missing={sorted(api_ids - raw_ids)[:3]}", fail="WARN")
    except Exception as e:
        check(S, "NWS alerts are real NWS alerts", "WARN", repr(e))

# MTA: parsed alert count vs raw entities in the subway feed
if not skip_if_offline("MTA subway alerts parsed", "transit_alerts"):
    try:
        raw = requests.get(config.MTA_ALERTS_URL.format(feed="subway-alerts"), timeout=15).json()
        n_raw = sum(1 for e in raw.get("entity", []) if e.get("alert"))
        parsed = feeds.parse_mta_alerts(raw, "subway")
        with_header = sum(1 for a in parsed if a["header"])
        expect(S, "MTA subway alerts parsed", n_raw == 0 or (parsed and with_header == len(parsed)),
               f"{n_raw} raw alerts, {len(parsed)} active within 24h, {with_header} with text")
    except Exception as e:
        check(S, "MTA subway alerts parsed", "WARN", repr(e))

# Permits: kept + skipped league games = raw rows in the same window
if not skip_if_offline("permitted events accounted for", "permitted_events"):
    pe = LIVE["permitted_events"]
    kept = len(pe["events"])
    expect(S, "permitted events accounted for", kept <= pe["rows"], f"{pe['rows']} raw rows, {kept} kept after dropping league games")

# Citi Bike: zone sums never exceed the citywide total
if not skip_if_offline("Citi Bike zone totals consistent", "citibike"):
    total = sum(s["ebikes"] for s in LIVE["citibike"]["stations"])
    zsum = sum((z["now"]["citibike"] or {}).get("ebikes", 0) for z in SIG["zones"])
    expect(S, "Citi Bike zone totals consistent", 0 <= zsum <= total, f"{zsum} e-bikes in tracked zones of {total} citywide")

# Street closures: every placed closure is active today
if not skip_if_offline("street closures active today", "street_closures"):
    today = datetime.now().date().isoformat()
    bad = [c for c in LIVE["street_closures"]["closures"] if not (c["start"] <= today <= c["end"])]
    expect(S, "street closures active today", not bad, f"{len(LIVE['street_closures']['closures'])} closures, {len(bad)} outside today")

## 5. Schema and invariants

In [ ]:
S = "schema"
client = webapp.app.test_client()
r = client.get("/api/v1/signals?hours=24")
expect(S, "GET /api/v1/signals", r.status_code == 200 or config.SIGNALS_API_KEY, f"status {r.status_code}")
if config.SIGNALS_API_KEY:
    r = client.get("/api/v1/signals?hours=24", headers={"X-API-Key": config.SIGNALS_API_KEY})
    expect(S, "API key accepted", r.status_code == 200)
    expect(S, "missing key rejected", client.get("/api/v1/signals").status_code == 401)
else:
    check(S, "API key configured", "WARN", "SIGNALS_API_KEY not set: fine on localhost, required on a public host")
D = r.get_json()

expect(S, "all tracked zones returned", len(D["zones"]) == len(zmod.zones()), f"{len(D['zones'])} zones")
expect(S, "24 hours per zone", all(len(z["hours"]) == 24 for z in D["zones"]))
ts = [datetime.fromisoformat(h["t"]) for h in D["city_hours"]]
expect(S, "hours are contiguous", all(b - a == timedelta(hours=1) for a, b in zip(ts, ts[1:])))

def keys(o):
    if isinstance(o, dict):
        for k, v in o.items():
            yield k; yield from keys(v)
    elif isinstance(o, list):
        for v in o:
            yield from keys(v)
banned = {"est", "lo", "hi", "pay_rate", "pay_per_engaged_hr", "utilization", "p25", "p75"}
found = banned & set(keys(D))
expect(S, "no earnings fields", not found and "$" not in json.dumps(D), f"found {sorted(found)}")

idx = [h["seasonal_trip_index"] for z in D["zones"] for h in z["hours"][:1] if h["seasonal_trip_index"] is not None]
expect(S, "seasonal index median is 1", idx and abs(statistics.median(idx) - 1) < 0.06, f"median {statistics.median(idx) if idx else None}")

# Alerts agree with the weather they were derived from
bad = []
for h in D["city_hours"]:
    w, types = h["weather"], {a["type"] for a in h["alerts"]}
    if not w:
        continue
    if ("heavy_rain" in types) != ((w["precip_mm"] or 0) >= config.ALERT_HEAVY_RAIN_MM):
        bad.append((h["t"], "heavy_rain"))
    if w["wind_mph"] is not None and ("high_wind" in types) != (w["wind_mph"] >= config.ALERT_HIGH_WIND_MPH):
        bad.append((h["t"], "high_wind"))
expect(S, "alerts consistent with weather", not bad, bad[:3])

air = [z["zone_id"] for z in D["zones"] if z["now"]["airport_delays"] is not None]
expect(S, "airport delays only on airport zones", set(air) <= {132, 138}, air)

phases_ok = True
for z in D["zones"]:
    for h in z["hours"]:
        for e in h["events"]:
            t, end = datetime.fromisoformat(h["t"]), datetime.fromisoformat(e["est_end"])
            if e["phase"] == "let_out" and not (end - timedelta(minutes=60) <= t <= end + timedelta(minutes=75)):
                phases_ok = False
expect(S, "event let-out windows line up", phases_ok)

matches = {a["match"] for z in D["zones"] for a in z["now"]["transit_alerts"]}
expect(S, "transit matches are valid", matches <= {"stop", "route", "terminal"}, sorted(matches))
expect(S, "provenance covers sections", all(k in json.dumps(D["provenance"]) for k in
       ["subway_entries", "permitted_events", "transit_alerts", "curb", "parking_tickets", "injury_crashes"]))

h = client.get("/api/v1/health", headers={"X-API-Key": config.SIGNALS_API_KEY} if config.SIGNALS_API_KEY else {}).get_json()
expect(S, "health endpoint", h.get("api_version") == config.SIGNALS_API_VERSION, h.get("sources"))
size_kb = len(r.data) / 1024
expect(S, "24h payload size reasonable", size_kb < 4096, f"{size_kb:.0f} KB", fail="WARN")

## 6. Public-records sanity

In [ ]:
S = "records"
ex = signals.EXTRAS.get()
NYC_BOX = (40.47, -74.27, 40.93, -73.68)
inside = lambda lat, lon: NYC_BOX[0] <= lat <= NYC_BOX[2] and NYC_BOX[1] <= lon <= NYC_BOX[3]

# Projection: a known DOT coordinate lands at W 35th St & Broadway
lat, lon = to_latlon(988000, 211900)
expect(S, "State Plane conversion", abs(lat - 40.74829) < 1e-4 and abs(lon + 73.98647) < 1e-4, f"{lat:.5f}, {lon:.5f}")

if not ex:
    check(S, "public-records checks", "SKIP", "no data/extras.json")
else:
    curb = (ex.get("curb") or {}).get("zones") or {}
    stands = [s for z in curb.values() for s in z["relief_stands"]]
    if "curb" not in ex:
        check(S, "curb data", "SKIP", "curb section didn't build (see build checks)")
    elif (ex.get("curb") or {}).get("relief_sign_rows", 0) == 0:
        check(S, "relief stands found", "WARN", "no signs matched RELIEF_STAND_PATTERNS: check sign wording in nfid-uabd")
    else:
        expect(S, "relief stands found", len(stands) > 0, f"{len(stands)} stands in tracked zones")
        expect(S, "relief stand coordinates inside NYC", all(inside(s["lat"], s["lon"]) for s in stands))
    if "curb" in ex:
        expect(S, "loading-zone blocks found", sum(z["loading_zone_blocks"] for z in curb.values()) > 0,
               f"{sum(z['loading_zone_blocks'] for z in curb.values())} blocks", fail="WARN")

    rid = (ex.get("ridership") or {}).get("zones") or {}
    if rid:
        # Commuter peak: weekday 8am entries should beat Sunday 8am. This also catches a day-of-week mix-up.
        wk = sum(v.get("1-8", 0) for v in rid.values()); sun = sum(v.get("6-8", 0) for v in rid.values())
        expect(S, "ridership: weekday 8am > Sunday 8am", wk > 1.5 * sun, f"Tue {wk:,.0f} vs Sun {sun:,.0f}")
        night = sum(v.get("1-3", 0) for v in rid.values())
        expect(S, "ridership: 8am > 3am", wk > 5 * max(night, 1), f"8am {wk:,.0f} vs 3am {night:,.0f}")
    else:
        check(S, "ridership", "SKIP", "ridership section didn't build (see build checks)")

    cr = (ex.get("crashes") or {}).get("zones") or {}
    if cr:
        tot = sum(sum(z["injury_crashes"].values()) for z in cr.values())
        expect(S, "crashes: plausible yearly count", 100 < tot < 60000, f"{tot:,.0f} injury crashes/yr in tracked zones")
    tk = (ex.get("tickets") or {}).get("precincts") or {}
    if tk:
        day = sum(v.get("10", 0) for v in tk.values()); night = sum(v.get("3", 0) for v in tk.values())
        expect(S, "tickets: 10am > 3am", day > night, f"{day:,.0f}/day at 10am vs {night:,.0f} at 3am")
    pz = (ex.get("precincts") or {}).get("zone_precinct") or {}
    expect(S, "every tracked zone has a precinct", len(pz) == len(zmod.zones()), f"{len(pz)}/{len(zmod.zones())}", fail="WARN")
    sub = ex.get("subway") or {}
    expect(S, "subway stations mapped", len(sub.get("stop_zone", {})) > 20, f"{len(sub.get('stop_zone', {}))} stops in tracked zones", fail="WARN")

## 7. Charts

In [ ]:
if not HAVE_PLOTS:
    print("skipped: pandas/matplotlib not installed")
else:
    names = {z["zone_id"]: z["name"] for z in D["zones"]}
    grid = pd.DataFrame({names[z["zone_id"]]: [h["seasonal_trip_index"] for h in z["hours"]] for z in D["zones"]},
                        index=[h["t"][11:16] for h in D["zones"][0]["hours"]]).T
    grid = grid.loc[grid.mean(axis=1).sort_values(ascending=False).index]
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(grid.values.astype(float), aspect="auto", cmap="viridis")
    ax.set_yticks(range(len(grid.index))); ax.set_yticklabels(grid.index, fontsize=7)
    ax.set_xticks(range(len(grid.columns))); ax.set_xticklabels(grid.columns, rotation=90, fontsize=7)
    ax.set_title("Seasonal trip index by zone, next 24 hours (1 = median zone)")
    fig.colorbar(im, ax=ax, shrink=0.6); plt.tight_layout(); plt.show()

    ex = signals.EXTRAS.get()
    rid = (ex.get("ridership") or {}).get("zones") or {}
    if rid:
        top = sorted(rid, key=lambda z: -sum(rid[z].values()))[:5]
        fig, ax = plt.subplots(figsize=(10, 4))
        for z in top:
            ax.plot(range(24), [rid[z].get(f"1-{h}", 0) for h in range(24)], label=names.get(int(z), z))
        ax.set_xlabel("hour (Tuesday)"); ax.set_ylabel("subway entries per hour"); ax.legend(fontsize=8)
        ax.set_title("Subway entries, busiest tracked zones"); plt.tight_layout(); plt.show()

    tk = (ex.get("tickets") or {}).get("precincts") or {}
    if tk:
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.bar(range(24), [sum(v.get(str(h), 0) for v in tk.values()) for h in range(24)])
        ax.set_xlabel("hour"); ax.set_ylabel("tickets per day"); ax.set_title("Parking tickets by hour, all precincts")
        plt.tight_layout(); plt.show()

## Summary

In [ ]:
counts = {s: sum(r["status"] == s for r in RESULTS) for s in ("PASS", "WARN", "FAIL", "SKIP")}
verdict = "FAIL" if counts["FAIL"] else ("PASS with warnings" if counts["WARN"] else "PASS")
print(f"Verdict: {verdict}   " + "  ".join(f"{k}={v}" for k, v in counts.items()))
report = {"run_at": datetime.now().isoformat(timespec="seconds"), "verdict": verdict, "counts": counts,
          "sources_live": ONLINE, "results": RESULTS}
(ROOT / "validation_report.json").write_text(json.dumps(report, indent=1))
print("wrote validation_report.json")
if HAVE_PLOTS:
    display(pd.DataFrame(RESULTS)[["status", "section", "check", "detail"]]
            .sort_values("status", key=lambda s: s.map({"FAIL": 0, "WARN": 1, "SKIP": 2, "PASS": 3})))
else:
    for r in RESULTS:
        print(f"{r['status']:4}  {r['section']:12} {r['check']}")